# C1.6 · High-concurrency detection engineering

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.5 · Emergent swarms and multi-agent proliferation](https://spbreed.github.io/cyber-commons/lessons/C1.5.html)**.

| | |
|---|---|
| Tools used | Sigma |

## What this lesson is

**What it covers.** Detection engineering for runtime-objective anomalies at high concurrency, choosing rules by the queue volume they add rather than by recall alone.

**Why a security engineer needs it.** A rule that fires on everything is worse than none, because it spends the attention the good rules need. Deployability is a measured property — firing volume against real history — not a matter of taste.

## 1 · The hook

A rule that flags an unauthorised runtime objective before compromise is worth a great deal, and the same rule firing on everything is worth less than nothing. Which one you built is decided by the volume it adds to the queue, not by its recall.

> **At CyberTravels.** The anomaly is a CyberTravels agent pursuing an objective its task never set, and the rules are scored against CyberTravels' own history so the queue cost is real.

## 2 · The framework

```
   five rules for one runtime-objective anomaly

   R1  fires 301x for 1 true positive     buries the queue   REJECT
   R2  fires   2x for 2 true positives    generalises        SHIP
   R3  100% precision on nothing useful                      REJECT

   every rule detects the anomaly. deployability is decided by the
   volume it adds, replayed against real history.
```

Detecting emergent behaviour at machine speed is a detection-engineering
problem. Semantic drift and runtime-objective anomalies are the signals, and the
trap is the same one every detection has: a rule that fires on everything is
worse than no rule, because it spends the attention the good rules need.

High-concurrency detection means generating candidate rules, then scoring them
against real history on the one property that decides deployability — the
firing volume — so a rule that flags an unauthorised objective before compromise
is kept and one that buries the queue is rejected with its numbers.

## 3 · The procedure, as a skill

Every candidate rule detects the anomaly. The skill replays each against CyberTravels' history and scores firing volume, so a rule that produces hundreds of alerts for one true positive is rejected with the number attached.

### The skill — [`skills/detection/detection-rule-deployability/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/detection-rule-deployability/SKILL.md)

```yaml
name: detection-rule-deployability
description: >-
  Score candidate detection rules on precision, recall and firing volume, and
  reject the ones no analyst could work regardless of how well they detect. Use
  when authoring detections with or without a model, or when a rule is proposed
  because it caught the incident.
allowed-tools: Read, Grep, Glob
```

# A rule that fires 301 times for one true positive is not a detection

Every candidate rule detects something. Deployability is a different property
and it is arithmetic: precision, recall, and how many times the rule fires per
day against real history. A rule failing on the third is rejected however good
the first two look, because it will be muted within a week.

## When to use this

Authoring detections, reviewing a model's proposed rules, and any time a rule is
proposed on the strength of catching one incident.

## Procedure

**1 — Replay each candidate against real history.** Not a sample chosen to
contain the incident — the actual period, including the quiet parts.

**2 — Compute precision, recall and volume.** Volume is the one people omit and
the one that decides whether the rule survives contact with an analyst.

**3 — Set a deployability bar before you look at the results.** Precision floor,
recall floor, and a maximum firings per day. Setting it afterwards means setting
it around the rule you like.

**4 — Reject the broad rules explicitly, with their numbers.** "Rejected: 301
firings for 1 true positive" is a sentence the author can act on; "too noisy" is
not.

**5 — Look at what survived, and what it depends on.** A high-precision rule
usually depends on a specific field being populated. Record that dependency —
it is the thing that will silently break the rule later.

## Example

**Input** — the fixture committed at the top of [`scripts/detection_rule_deployability.py`](scripts/detection_rule_deployability.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
history: 522 events, 2 true positives
rule                                 alerts   prec  recall  alerts/TP
----------------------------------------------------------------------
R1 any http_get by an agent             301  0.003   0.500      301.0
R2 http_get to a non-github host          1  1.000   0.500        1.0
R3 link-local address                     1  1.000   0.500        1.0
R4 any failed action                     20  0.000   0.000        inf
R5 credential path OR link-local          2  1.000   1.000        1.0
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "history": {"events": 0, "period_days": 0, "true_positives": 0},
  "candidates": [{"name": "str", "fires": 0, "tp": 0, "precision": 0.0, "recall": 0.0,
                  "per_day": 0.0, "verdict": "deploy|reject", "why": "str"}],
  "bar": {"precision": 0.0, "recall": 0.0, "max_per_day": 0},
  "dependencies": [{"rule": "str", "requires_field": "str"}]
}
```

## Failure modes

- **Replaying against a period chosen to contain the incident.** Volume becomes
  meaningless.
- **Setting the bar after seeing the results.** That is choosing a winner.
- **Deploying a rule with an unrecorded field dependency.** It fails silently
  when the field stops being populated.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/detection-rule-deployability/scripts/detection_rule_deployability.py
SCRIPT = "skills/detection/detection-rule-deployability/scripts/detection_rule_deployability.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

All candidates detect the anomaly, but their firing volumes differ by orders of magnitude, and the deployable one is chosen by the volume it would add to the queue rather than by recall alone.

## Your turn

Take a detection you are proud of and compute how many times it fired last month against how many were true. If you cannot, the rule is unmeasured, which is the same as untuned.

---

**Next → [C1.7 · Triaging the non-deterministic swarm](https://spbreed.github.io/cyber-commons/lessons/C1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*